# Stage 1: Chốt schema + refactor skeleton

## 1. Load Bronze Dataset

In [ ]:
from pathlib import Path

import fiftyone as fo

from z_photos.conf import Config
from z_photos.datasets import BronzeField, build_bronze_dataset

cfg = Config(
    data_root=Path("../data"),
    seed=42,
)

bronze_dataset = build_bronze_dataset("z_photos-bronze", bronze_dir=cfg.bronze_dir)
print(f"Bronze dataset size: {len(bronze_dataset)}")

Bronze dataset size: 321


## 2. Refactor: Update Bronze schema with flags

In [ ]:
# Add missing flags to Bronze layer to align with Stage 1 contract
def init_boolean_flag(dataset, field_name: str, default: bool = False):
    if not dataset.has_sample_field(field_name):
        dataset.add_sample_field(field_name, fo.BooleanField)
        dataset.set_values(field_name, [default] * len(dataset))


init_boolean_flag(bronze_dataset, BronzeField.IS_EXACT_DUP)
init_boolean_flag(bronze_dataset, BronzeField.IS_NEAR_DUP)
init_boolean_flag(bronze_dataset, BronzeField.IS_LEAKY)

# Currently, `main.ipynb` marked leaky samples using tags
# Let's port that knowledge over to the newly established boolean field if available
for sample in bronze_dataset:
    changed = False
    if "leaky" in sample.tags:
        sample[BronzeField.IS_LEAKY] = True
        sample.tags.remove("leaky")
        changed = True

    if changed:
        sample.save()

print("Bronze schema updated.")

Bronze schema updated.


## 3. Clone for Silver and Gold Views

In [ ]:
from fiftyone import ViewField


def get_or_create_clone(dataset, view, clone_name: str):
    if clone_name in fo.list_datasets():
        # Clean up existing ones for this experiment if we want to rebuild
        fo.delete_dataset(clone_name)
    return view.clone(clone_name, persistent=True)


# 1. Silver Train (from train tag)
silver_train_view = bronze_dataset.match_tags("train")
silver_train = get_or_create_clone(
    bronze_dataset, silver_train_view, "z_photos-silver-train"
)
print(f"Silver Train created: {len(silver_train)} samples")

# 2. Gold Eval Clean (from test tag, excluding leaky)
gold_eval_clean_view = bronze_dataset.match_tags("test").match(
    ~ViewField(BronzeField.IS_LEAKY)
)
gold_eval_clean = get_or_create_clone(
    bronze_dataset, gold_eval_clean_view, "z_photos-gold-eval-clean"
)
print(f"Gold Eval Clean created: {len(gold_eval_clean)} samples")

Silver Train created: 261 samples
Gold Eval Clean created: 50 samples
